In [7]:
import pandas as pd

In [12]:
file_path = r"F:\机场噪音\副本Final_Variable_2021_noise_Regression-补全.xlsx"
year = 2021
data_path = rf"F:\机场噪音\Final_Analysis_矫正\国家级\L0_Global_Noise_Impact_{year}_Wide.csv"

In [13]:
cat_df = pd.read_excel(file_path, engine='openpyxl')
df = pd.read_csv(data_path)

In [14]:
cat_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175 entries, 0 to 174
Data columns (total 52 columns):
 #   Column                                                                    Non-Null Count  Dtype  
---  ------                                                                    --------------  -----  
 0   gaul0_name                                                                175 non-null    object 
 1   North_South                                                               175 non-null    object 
 2   Continent_six                                                             175 non-null    object 
 3   total_gdp                                                                 175 non-null    int64  
 4   total_pop                                                                 175 non-null    float64
 5   SEL_night_202110_night_40dB                                               175 non-null    float64
 6   SEL_night_202110_night_45dB                                       

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 26 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   gaul0_name                           263 non-null    object 
 1   total_pop                            263 non-null    float64
 2   SEL_night_202110_night_40dB          263 non-null    float64
 3   SEL_night_202110_night_45dB          263 non-null    float64
 4   SEL_night_202110_night_50dB          263 non-null    float64
 5   SEL_night_202110_night_55dB          263 non-null    float64
 6   SEL_night_202110_night_60dB          263 non-null    float64
 7   SEL_night_202110_night_65dB          263 non-null    float64
 8   SEL_oneday_202110_oneday_45dB        263 non-null    float64
 9   SEL_oneday_202110_oneday_50dB        263 non-null    float64
 10  SEL_oneday_202110_oneday_55dB        263 non-null    float64
 11  SEL_oneday_202110_oneday_60dB   

In [ ]:
import pandas as pd

# 1. 基础配置
years = [2021, 2022, 2023]
file_path = r"F:\机场噪音\副本Final_Variable_2021_noise_Regression-补全.xlsx"
cat_df = pd.read_excel(file_path, engine='openpyxl')[['gaul0_name', 'North_South', 'Continent_six']]

all_years_data = []

# 2. 循环处理每一年的数据
for year in years:
    # 动态读取对应年份的数据
    data_path = rf"F:\机场噪音\Final_Analysis_矫正\国家级\L0_Global_Noise_Impact_{year}_Wide.csv"
    try:
        df_year = pd.read_csv(data_path)
    except FileNotFoundError:
        print(f"跳过 {year}，未找到文件: {data_path}")
        continue
    
    # 合并标签
    # 修改合并逻辑，只保留两个表共有的国家
    merged = pd.merge(df_year, cat_df, on='gaul0_name', how='inner')

    # 检查一下丢弃了多少国家
    dropped_count = len(df_year) - len(merged)
    if dropped_count > 0:
        print(f"{year}年有 {dropped_count} 个国家因为不在分类表中而被剔除。")
    
    # 确定暴露人口列 (排除含有 ratio 的比例列，只保留原始人数列)
    pop_columns = [col for col in df_year.columns if 'dB' in col and 'ratio' not in col]
    
    # --- A. 准备三个维度的聚合列表 ---
    # 维度1: 大洲 (Continent_six)
    # 维度2: 南北方 (North_South)
    # 维度3: 全球 (Global) - 构造一个虚拟列进行全量求和
    temp_merged = merged.copy()
    temp_merged['Global'] = 'World' 
    
    dimensions = {
        'Continent_six': 'Continent',
        'North_South': 'North_South',
        'Global': 'Global'
    }

    for group_col, dim_name in dimensions.items():
        summary = temp_merged.groupby(group_col).agg({
            'total_pop': 'sum',
            **{col: 'sum' for col in pop_columns}
        })
        
        # 计算该维度下的暴露率
        for col in pop_columns:
            summary[f'{col}_rate'] = summary[col] / summary['total_pop']
        
        # 整理索引，统一列名以方便 concat
        summary = summary.reset_index().rename(columns={group_col: 'Category'})
        summary['Dimension'] = dim_name
        summary['Year'] = year
        
        all_years_data.append(summary)

# 3. 合并所有维度和年份
final_raw = pd.concat(all_years_data, ignore_index=True)

# 4. 转换为透视表：Year 在最顶层，指标在第二层
final_pivot = final_raw.pivot_table(
    index=['Dimension', 'Category'], 
    columns='Year'
)

# 5. 调整列顺序：让年份对齐，指标在其下方
# reorder_levels([1, 0]) 将 Year 移到第一层级，sort_index 确保年份按顺序排列
final_pivot = final_pivot.reorder_levels([1, 0], axis=1).sort_index(axis=1)

# 6. 保存并预览
# final_pivot.to_excel("Global_Noise_2021-2023_MultiLevel.xlsx")
print("数据统计完成！前几行预览：")
print(final_pivot.head())

2021年有 88 个国家因为不在分类表中而被剔除。
2022年有 88 个国家因为不在分类表中而被剔除。
2023年有 88 个国家因为不在分类表中而被剔除。
数据统计完成！前几行预览：
Year                                           2021  \
                        SEL_night_202110_night_40dB   
Dimension Category                                    
Continent Africa                       6.356260e+06   
          Asia                         4.735340e+07   
          Europe                       1.425806e+07   
          North America                4.250773e+07   
          Oceania                      4.528064e+05   

Year                                                      \
                        SEL_night_202110_night_40dB_rate   
Dimension Category                                         
Continent Africa                                0.005190   
          Asia                                  0.010522   
          Europe                                0.019101   
          North America                         0.072039   
          Oceania                           

In [ ]:
final_pivot

Year                                             2021  \
                          SEL_night_202110_night_40dB   
Dimension   Category                                    
Continent   Africa                       6.356260e+06   
            Asia                         4.735340e+07   
            Europe                       1.425806e+07   
            North America                4.250773e+07   
            Oceania                      4.528064e+05   
            South America                8.938383e+06   
Global      World                        1.198666e+08   
North_South Global North                 5.677345e+07   
            Global South                 6.309318e+07   

Year                                                        \
                          SEL_night_202110_night_40dB_rate   
Dimension   Category                                         
Continent   Africa                                0.005190   
            Asia                                  0.010522   
            Europe                                0.019101   
            North America                         0.072039   
            Oceania                               0.010915   
            South America                         0.021005   
Global      World                                 0.015922   
North_South Global North                          0.042006   
            Global South                          0.010214   

Year                                                   \
                          SEL_night_202110_night_45dB   
Dimension   Category                                    
Continent   Africa                       1.464306e+06   
            Asia                         2.059572e+07   
            Europe                       4.892409e+06   
            North America                2.026778e+07   
            Oceania                      1.354837e+05   
            South America                4.217846e+06   
Global      World                        5.157354e+07   
North_South Global North                 2.505953e+07   
            Global South                 2.651401e+07   

Year                                                        \
                          SEL_night_202110_night_45dB_rate   
Dimension   Category                                         
Continent   Africa                                0.001196   
            Asia                                  0.004577   
            Europe                                0.006554   
            North America                         0.034348   
            Oceania                               0.003266   
            South America                         0.009912   
Global      World                                 0.006850   
North_South Global North                          0.018541   
            Global South                          0.004292   

Year                                                   \
                          SEL_night_202110_night_50dB   
Dimension   Category                                    
Continent   Africa                       2.624797e+05   
            Asia                         7.463249e+06   
            Europe                       1.567679e+06   
            North America                7.580926e+06   
            Oceania                      3.546068e+04   
            South America                1.656977e+06   
Global      World                        1.856677e+07   
North_South Global North                 9.045214e+06   
            Global South                 9.521557e+06   

Year                                                        \
                          SEL_night_202110_night_50dB_rate   
Dimension   Category                                         
Continent   Africa                                0.000214   
            Asia                                  0.001658   
            Europe                                0.002100   
            North America                         0.012848   
            Oceania  

In [ ]:
import pandas as pd
from openpyxl.utils import get_column_letter

# 1. 设置保存路径
output_path = r"F:\机场噪音\Global_Noise_Exposure_2021-2023_Summary.xlsx"

# 2. 保存并处理格式
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    final_pivot.to_excel(writer, sheet_name='Noise_Exposure_Stats')
    
    worksheet = writer.sheets['Noise_Exposure_Stats']
    
    # 调整列宽的鲁棒性写法
    # 使用 enumerate 避开对 cell 属性的直接依赖
    for i, col in enumerate(worksheet.columns, 1):
        max_length = 0
        column_letter = get_column_letter(i) # 直接根据索引生成列字母 A, B, C...
        
        for cell in col:
            try:
                if cell.value:
                    val_len = len(str(cell.value))
                    if val_len > max_length:
                        max_length = val_len
            except:
                pass
        
        # 设置列宽，最小 10，最大 40，防止某些列过宽
        adjusted_width = min(max(max_length + 2, 10), 40)
        worksheet.column_dimensions[column_letter].width = adjusted_width

print(f"✨ 文件已成功保存至: {output_path}")

✨ 文件已成功保存至: F:\机场噪音\Global_Noise_Exposure_2021-2023_Summary.xlsx


: 